In [ ]:
import numpy as np
import numpy.typing as npt

try:
    import cupy as cp
    import cupyx

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        scatter_add = cupyx.scatter_add
        device = "cuda"
    else:
        raise Exception("CUDA is not available")
except (ImportError, Exception):
    xp = np
    scatter_add = np.add.at
    device = "cpu"

device

'cuda'

In [2]:
class ReLU:
    _inputs: npt.NDArray = None

    def __init__(self):
        pass

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        self._inputs = inputs

        return xp.maximum(0, inputs)

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return xp.where(self._inputs > 0, grad, 0)

In [3]:
class Softmax:
    _outputs: npt.NDArray = None

    def __init__(self):
        pass

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        max_logit = xp.max(inputs, axis=-1, keepdims=True)

        self._outputs = xp.exp(inputs - max_logit) / xp.sum(
            xp.exp(inputs - max_logit), axis=-1, keepdims=True
        )

        return self._outputs

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return grad * self._outputs - self._outputs * xp.sum(
            grad * self._outputs, axis=-1, keepdims=True
        )

In [4]:
class PositionalEncoding:
    def __init__(self, embedding_length: int, scale=1.0):
        self._embedding_length = embedding_length
        self._scale = scale

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        T = inputs.shape[1]

        pos = xp.arange(0, T, dtype=int)[:, xp.newaxis]

        denominator = (
             xp.arange(0, self._embedding_length) // 2 * 2
        ) / self._embedding_length
        denominator = 1 / (10_000**denominator)
        denominator = denominator[xp.newaxis, ...]

        inner = pos * denominator
        pe_sin = xp.sin(inner)
        pe_cos = xp.cos(inner)

        pe_sin[:, xp.arange(1, self._embedding_length, 2)] = 0
        pe_cos[:, xp.arange(0, self._embedding_length, 2)] = 0

        positional_encoding = pe_sin + pe_cos

        return inputs + positional_encoding[xp.newaxis, ...] * self._scale

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return grad

In [ ]:
class EmbeddingLayer:
    _inputs: npt.NDArray = None

    def __init__(self, vocab_size, embedding_dim):
        scale = 1.0 / math.sqrt(embedding_dim)
        
        self.embeddings = xp.random.randn(vocab_size, embedding_dim) * scale
        self.embeddings_grad = xp.zeros(self.embeddings.shape)

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        self._inputs = inputs
        return self.embeddings[inputs]
    
    def backward(self, grad: npt.NDArray):
        scatter_add(self.embeddings_grad, self._inputs, grad)

In [ ]:
import math


class Linear:
    _inputs: npt.NDArray = None

    def __init__(self, in_features: int, out_features: int, include_bias=True):
        limit = math.sqrt(6 / (in_features + out_features))

        self.weights = xp.random.uniform(-limit, limit, (out_features, in_features))
        self.bias = xp.zeros(out_features) if include_bias else None

        self.weights_grad = xp.zeros(self.weights.shape)
        self.bias_grad = xp.zeros(self.bias.shape) if include_bias else None

        self._include_bias = include_bias

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        self._inputs = inputs

        out = xp.matmul(inputs, self.weights)

        if self._include_bias:
            out += self.bias

        return out

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        inputs_flat = self._inputs.reshape(-1, self._inputs.shape[-1])
        grad_flat = grad.reshape(-1, grad.shape[-1])

        dL_dA = xp.matmul(grad, self.weights.T)
        dL_dW = xp.matmul(inputs_flat.T, grad_flat)

        if self._include_bias:
            dL_dB = xp.sum(grad_flat, axis=0, keepdims=True)
            self.bias_grad += dL_dB

        self.weights_grad += dL_dW

        return dL_dA

In [7]:
class FeedForwardNetwork:
    def __init__(self, input_units, hidden_units, output_units):
        self.linear1 = Linear(input_units, hidden_units)
        self.activation = ReLU()
        self.linear2 = Linear(hidden_units, output_units)

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        x = self.linear1(inputs)
        x = self.activation(x)
        x = self.linear2(x)

        return x

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        grad = self.linear2.backward(grad)
        grad = self.activation.backward(grad)
        grad = self.linear1.backward(grad)

        return grad